# Preprocessing: prepare recordins for training

Purpose
-------
Given raw files produced by `data/recording_support/recording_script.pt-BR.md` this notebook converts raw WAVs into consistent, training-ready audio:

- convert to mono, target sample rate (e.g. 22050);
- trim long leading/trailing silence; 
- normalize loudness;
- export 16-bit PCM WAVs into `data/processed/wavs/`;
- produce `data/processed/metadata.csv` (LJSpeech style: filename|transcriptions|speaker_id)

Run this after you place files in data/raw/recordings/. Work on a small subset first.

In [ ]:
from pathlib import Path

import soundfile as sf
from IPython.display import Audio, display

RAW_DIR = Path("data/raw/recordings")
OUTPUT_DIR = Path("data/processed/wavs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 22050
MIN_DURATION = 0.2  # seconds, drop short files
MAX_DURATION = 20.0  # seconds, split or drop long files

EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}


def list_audio(folder=RAW_DIR, number_of_files=10):
    if not folder.exists():
        print(f"{folder} does not exist. Put raw files there (from recording script).")
        return []

    files = sorted([p for p in folder.rglob("*") if p.suffix.lower() in EXTENSIONS])
    for idx, file in enumerate(files[:number_of_files], 1):
        try:
            info = sf.info(str(file))
            duration = (
                info.frames / info.samplerate
                if info.frames and info.samplerate
                else None
            )
            print(
                f"{idx}. {file} - sample_rate={info.samplerate}, channels={info.channels}, duration={duration:.3f}s"
            )
        except Exception as error:
            print(f"{idx}. {file} - ERROR: {error}")
    return files


files = list_audio()

if files:
    print("Playing first file:")
    display(Audio(str(files[0])))

## Processing function

Read -> mono -> resample -> trim silence -> normalize -> save 16-bit WAV

Why?
    - Mono & resamble: models expect a single consistent sample rate & channel count;
    - Trim silence: reduces wasted frames and stabilizes aligment;
    - RMS normalize: keeps loudness consistent across files;
    - Duration checks: remove or flag extremely short/long utterances before training.

In [ ]:
import numpy as np

try:
    import librosa
except Exception:
    librosa = None


def to_mono(audio):
    if audio.ndim == 1:
        return audio
    return np.mean(audio, axis=1)


def normalize_rms(wav, target_db=-20.0, epsilon=1e-9):
    rms = np.sqrt(np.mean(wav**2) + epsilon)
    target_lin = 10.0 ** (target_db / 20.0)
    scale = target_lin / rms
    return wav * scale


def trim_silence(wav, sample_rate, top_db=40):
    if librosa is not None:
        intervals = librosa.effects.split(wav, top_db=top_db)
        if intervals.size == 0:
            return wav
        start, end = intervals[0, 0], intervals[-1, 1]
        return wav[start:end]

    frame_len = int(0.02 * sample_rate) or 1
    frame_step = frame_len
    energy = np.array(
        [
            np.mean(np.abs(wav[idx : idx + frame_len]))
            for idx in range(0, len(wav), frame_step)
        ]
    )
    thresh = np.max(energy) * 0.01
    nonzero = np.where(energy > thresh)[0]
    if nonzero.size == 0:
        return wav
    start = max(0, nonzero[0] * frame_step)
    end = min(len(wav), (nonzero[-1] + 1) * frame_step)
    return wav[start:end]


def resample_if_needed(wav, original_sample_rate, target_sample_rate):
    if original_sample_rate == target_sample_rate:
        return wav

    if librosa is not None:
        return librosa.resample(
            wav, orig_sr=original_sample_rate, target_sr=target_sample_rate
        )

    duration = len(wav) / original_sample_rate

    new_length = int(round(duration * target_sample_rate))
    if new_length <= 0:
        return np.array([], dtype=wav.dtype)

    old_index = np.linspace(0, len(wav) - 1, num=len(wav))
    new_index = np.linspace(0, len(wav) - 1, num=new_length)

    return np.interp(new_index, old_index, wav).astype(wav.dtype)


def process_file(
    in_path: Path, output_dir: Path, target_sample_rate=TARGET_SAMPLE_RATE
):

    data, sample_rate = sf.read(str(in_path), always_2d=True)

    wav = to_mono(data)
    wav = resample_if_needed(
        wav, original_sample_rate=sample_rate, target_sample_rate=target_sample_rate
    )
    sample_rate = target_sample_rate

    wav = trim_silence(wav, sample_rate, top_db=40)
    wav = normalize_rms(wav, target_db=-20.0)

    duration = len(wav) / sample_rate if sample_rate > 0 else 0.0

    if duration < MIN_DURATION:
        raise ValueError(f"Too short ({duration:.3f}s)")
    if duration > MAX_DURATION:
        raise ValueError(f"Too long ({duration:.3f}s)")

    output_path = output_dir / in_path.name
    sf.write(str(output_path), wav, sample_rate, subtype="PCM_16")
    return output_path, sample_rate, duration

## Test processing

Test processing on a single file

In [ ]:
if not files:
    print('No files to process. Place recordings in "data/raw/recordings"')
else:
    preview_file = files[0]
    try:
        output_path, sample_rate, duration = process_file(preview_file, OUTPUT_DIR)
        print("Processed preview ->", output_path, sample_rate, f"{duration:.3f}s")
        display(Audio(str(output_path)))
    except Exception as error:
        print("Preview processing failed:", error)

## Filling metadata.csv

- The training data pipeline expects a mapping: `filename|transcriptions|speaker_id` (LJSpeech style);
- The notebook will generate a metadata template with empty transcriptions preserving your recording script filenames;
- After preprocessing, open `data/processed/metadata.csv` and fill the transcription column with normalized text (use the normalization helpers below);
- Do NOT commit raw or processed audio to Git, commit only the small metadata CSV.

In [ ]:
from csv import writer

SPEAKER_ID = "vinigalantine"

processed = []
skipped = []

for idx, file in enumerate(list_audio(), start=1):
    try:
        output_path, sample_rate, duration = process_file(file, OUTPUT_DIR)
        file_name = output_path.name
        processed.append((file_name, "", SPEAKER_ID))
        print("OK:", file_name, f"{duration:.2f}s")
    except Exception as error:
        skipped.append((file.name, str(error)))
        print("SKIP:", file.name, "->", error)

metadata_path = Path("data/processed/metadata.csv")
with open(metadata_path, "w", encoding="utf-8", newline="") as file:
    csv_writer = writer(file, delimiter="|")
    for row in processed:
        csv_writer.writerow(row)

print("Wrote metadata template to", metadata_path)
if skipped:
    print("Skipped files (inspect):")
    for name, error in skipped:
        print(" ", name, "->", error)

## Transcript normalization (why and how)

Why:
    - Training needs consistent, predictable text. We should normalize numbers, dates, abbreviations, punctuation and remove unwanted symbols;
    - Inconsistent transcripts lead to inconsistent pronunciations.

Basic rules (pt-BR):
    - Lowercase, remove unwanted symbols;
    - Expand numbers/dates/currency into words (use `num2words` for Portuguese or a custom rule set);
    - Normalize punctuation: keep only characters that afftect pornunciation (commas, question marks);
    - Match the exact spoken text

Example: "R$ 1.234,56" -> "mil duzentos e trinta e quatro reais e cinquenta e seis centavos"

In [2]:
import re

try:
    from num2words import num2words
except Exception:
    num2words = None


def replace_num(num):
    if num2words:
        return num2words(int(num.group(0)), lang="pt_BR")

    return num.group(0)


def simple_normalization(original_text):
    text = original_text.strip()
    text = text.replace("–", "-")
    text = re.sub(r"\s+", " ", text)
    text = text.lower()
    text = re.sub(r"\b\d+\b", replace_num, text)
    text = re.sub(r"[^a-zà-ú0-9\-\.,\?!:; ]", "", text)
    return text


normalization_tests = ["R$ 1.234,56", "21/09/2025", "Meu número é 0819999550"]

for normalization_test in normalization_tests:
    print(normalization_test, "->", simple_normalization(normalization_test))

R$ 1.234,56 -> r um.duzentos e trinta e quatro,cinquenta e seis
21/09/2025 -> vinte e umnovedois mil e vinte e cinco
Meu número é 0819999550 -> meu número é oitocentos e dezenove milhões, novecentos e noventa e nove mil, quinhentos e cinquenta


## How to fill metadata safely

- Open `data/processed/metadata.csv` in a text editor or spreadsheet;
- For each line, put the normalized transcription in the 2nd column (separator `|`). Example:
    `frase_001.wav|Olá, meu nome é Vinícius|vinigalantine`
- After filling, run `scripts/validate_dataset.py` or re-run `notebooks/02_data_audit.ipynb` to verify sample rate/duration match metadata;
